## Bronze Layer - Raw Employee Data
**Purpose**:Load incoming batch and perform MERGE (Upsert) into Bronze table.

## ✅ Delta Tables

- So first ensure your table is Delta:
- Convert table (run once)

In [0]:
%skip
%sql
CONVERT TO DELTA rohan_catalog.benefits_demo.employee_data;

## Batch Data (Mix of New + Updates)
Updates:
- rohan_catalog.benefits_demo.employee_data

In [0]:
%skip
new_data = [
    (110, "Anusha", 27, "F", "Finance", 75000, "Y", "Y", "Active"),  # updated
    (111, "Vikram", 42, "M", "IT", 90000, "Y", "N", "Active"),       # updated
    (120, "Rahul", 30, "M", "IT", 60000, "N", "N", "Active"),        # new
    (121, "Keerthi", 35, "F", "HR", 65000, "Y", "N", "Active")       # new
]

# Create Batch DataFrame

columns = ["EmpID", "Name", "Age", "Gender", "Department", "Salary", "Smoking", "Diabetic", "EnrollmentStatus"]
new_df = spark.createDataFrame(new_data, columns)

## Step 1: Read source CSV file(s) from Unity Catalog Volume

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv")

## Step 2: Add metadata columns (best practice)

In [0]:
from pyspark.sql.functions import current_timestamp, col

df = df.withColumn("LoadTimestamp", current_timestamp()) \
       .withColumn("SourceFile", col("_metadata.file_path"))

In [0]:
print("Incoming batch data:")
display(df)

Incoming batch data:


EmpID,Name,Age,Gender,Department,Salary,Smoking,Diabetic,EnrollmentStatus,LoadTimestamp,SourceFile
101,Rohan,29,M,IT,70000,Y,N,Active,2026-04-05T13:38:18.764Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
104,Pooja,30,F,IT,58000,N,N,Active,2026-04-05T13:38:18.764Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
106,Kiran,39,M,HR,68000,N,N,Active,2026-04-05T13:38:18.764Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
107,Anjali,31,F,Finance,72000,Y,N,Active,2026-04-05T13:38:18.764Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
108,Manoj,44,M,Operations,85000,N,Y,Active,2026-04-05T13:38:18.764Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
109,Divya,27,F,HR,54000,N,N,Inactive,2026-04-05T13:38:18.764Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
110,Tejas,36,M,IT,76000,Y,Y,Active,2026-04-05T13:38:18.764Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
111,Lavanya,33,F,Finance,69000,N,N,Active,2026-04-05T13:38:18.764Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv


## Below step executed once,
- Because Bronze table(employee_data) originally had only business columns like: 
    EmpID, Name, Age, Gender, Department, Salary, Smoking, Diabetic, EnrollmentStatus
- But in Bronze notebook, you later started loading extra metadata columns: 
    LoadTimestamp, SourceFile

In [0]:
%skip
%sql
ALTER TABLE rohan_catalog.benefits_demo.employee_data
ADD COLUMNS (
  LoadTimestamp TIMESTAMP,
  SourceFile STRING
);

## Step 3: Create temp view for MERGE

In [0]:
df.createOrReplaceTempView("updates")

## Step 4: MERGE into Bronze table

In [0]:
%sql
MERGE INTO rohan_catalog.benefits_demo.employee_data AS target
USING updates AS source
ON target.EmpID = source.EmpID

WHEN MATCHED THEN
UPDATE SET
    target.Name = source.Name,
    target.Age = source.Age,
    target.Gender = source.Gender,
    target.Department = source.Department,
    target.Salary = source.Salary,
    target.Smoking = source.Smoking,
    target.Diabetic = source.Diabetic,
    target.EnrollmentStatus = source.EnrollmentStatus,
    target.LoadTimestamp = source.LoadTimestamp,
    target.SourceFile = source.SourceFile

WHEN NOT MATCHED THEN
INSERT (
    EmpID,
    Name,
    Age,
    Gender,
    Department,
    Salary,
    Smoking,
    Diabetic,
    EnrollmentStatus,
    LoadTimestamp,
    SourceFile
)
VALUES (
    source.EmpID,
    source.Name,
    source.Age,
    source.Gender,
    source.Department,
    source.Salary,
    source.Smoking,
    source.Diabetic,
    source.EnrollmentStatus,
    source.LoadTimestamp,
    source.SourceFile
)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
8,8,0,0


## Also can write with Spark

In [0]:
%skip
spark.sql("""
MERGE INTO rohan_catalog.benefits_demo.employee_data AS target
USING updates AS source
ON target.EmpID = source.EmpID

WHEN MATCHED THEN UPDATE SET
    target.Name = source.Name,
    target.Age = source.Age,
    target.Gender = source.Gender,
    target.Department = source.Department,
    target.Salary = source.Salary,
    target.Smoking = source.Smoking,
    target.Diabetic = source.Diabetic,
    target.EnrollmentStatus = source.EnrollmentStatus,
    target.LoadTimestamp = source.LoadTimestamp,
    target.SourceFile = source.SourceFile

WHEN NOT MATCHED THEN INSERT (
    EmpID,
    Name,
    Age,
    Gender,
    Department,
    Salary,
    Smoking,
    Diabetic,
    EnrollmentStatus,
    LoadTimestamp,
    SourceFile
)
VALUES (
    source.EmpID,
    source.Name,
    source.Age,
    source.Gender,
    source.Department,
    source.Salary,
    source.Smoking,
    source.Diabetic,
    source.EnrollmentStatus,
    source.LoadTimestamp,
    source.SourceFile
)
""")

print("Bronze layer updated successfully!")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Step 5: Validate Bronze table

In [0]:
%sql
SELECT * FROM rohan_catalog.benefits_demo.employee_data ORDER BY EmpID;

EmpID,Name,Age,Gender,Department,Salary,Smoking,Diabetic,EnrollmentStatus,LoadTimestamp,SourceFile
101,Rohan,29,M,IT,70000,Y,N,Active,2026-04-05T13:39:26.381Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
102,Sneha,34,F,HR,60000,N,Y,Active,null,null
103,Amit,45,M,Finance,80000,Y,Y,Inactive,null,null
104,Pooja,30,F,IT,58000,N,N,Active,2026-04-05T13:39:26.381Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
105,Ravi,50,M,Operations,90000,Y,Y,Active,null,null
106,Kiran,39,M,HR,68000,N,N,Active,2026-04-05T13:39:26.381Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
107,Anjali,31,F,Finance,72000,Y,N,Active,2026-04-05T13:39:26.381Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
108,Manoj,44,M,Operations,85000,N,Y,Active,2026-04-05T13:39:26.381Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
109,Divya,27,F,HR,54000,N,N,Inactive,2026-04-05T13:39:26.381Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
110,Tejas,36,M,IT,76000,Y,Y,Active,2026-04-05T13:39:26.381Z,dbfs:/Volumes/rohan_catalog/benefits_demo/input_files/employee_batch_3.csv
